# 01 — Environment Setup

Imports, device detection, and path configuration.  
Run this notebook first; the cells below set variables (`DATA_DIR`, `SAVE_DIR`, `DEVICE`) used by the training and evaluation notebooks.

In [ ]:
import sys, os
sys.path.insert(0, '..')  # make src/ importable

import warnings, random
import numpy as np
import torch
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

from src.config import DEVICE
print('PyTorch :', torch.__version__)
print('Device  :', DEVICE)

In [ ]:
import zipfile, shutil

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    ZIP_PATH_IN_DRIVE = '/content/drive/MyDrive/good_data.zip'   # <-- edit if needed
    LOCAL_DATA_ROOT    = '/content'                              # extraction target
    LOCAL_DATA_DIR     = os.path.join(LOCAL_DATA_ROOT, 'good_data')

    assert os.path.exists(ZIP_PATH_IN_DRIVE), (
        f'Could not find {ZIP_PATH_IN_DRIVE}. '
        f'Update ZIP_PATH_IN_DRIVE to point at good_data.zip in your Drive.'
    )

    already_extracted = os.path.isdir(LOCAL_DATA_DIR) and len(os.listdir(LOCAL_DATA_DIR)) > 0
    if already_extracted:
        print(f'✓ {LOCAL_DATA_DIR} already populated — skipping extraction.')
    else:
        print(f'Extracting {ZIP_PATH_IN_DRIVE} -> {LOCAL_DATA_ROOT} ...')
        with zipfile.ZipFile(ZIP_PATH_IN_DRIVE, 'r') as zf:
            zf.extractall(LOCAL_DATA_ROOT)
        print('Done.')

    # zip files sometimes wrap contents in an extra top-level folder
    # (e.g. good_data.zip -> good_data/good_data/...). Flatten that case.
    inner = os.path.join(LOCAL_DATA_DIR, 'good_data')
    if os.path.isdir(inner):
        print('Flattening nested good_data/good_data -> good_data ...')
        for item in os.listdir(inner):
            shutil.move(os.path.join(inner, item), os.path.join(LOCAL_DATA_DIR, item))
        os.rmdir(inner)

    DATA_DIR = LOCAL_DATA_DIR
    SAVE_DIR = '/content/drive/MyDrive/clustering_v2'   # checkpoints saved back to Drive
else:
    # Local / non-Colab fallback — edit these for your machine.
    DATA_DIR = '../good_data'
    SAVE_DIR = '../clustering_v2'

print(f'DATA_DIR = {DATA_DIR}')
print(f'SAVE_DIR = {SAVE_DIR}')

In [ ]:
# Checkpoint, log, and data paths — derived from DATA_DIR / SAVE_DIR
os.makedirs(SAVE_DIR, exist_ok=True)

IL_SAVE     = os.path.join(SAVE_DIR, 'transformer_imitation_v2.pt')
IL_LOG      = os.path.join(SAVE_DIR, 'il_training_log.csv')

TRAIN_DIR   = os.path.join(DATA_DIR, 'synthetic_train')
TEST_DIR    = os.path.join(DATA_DIR, 'synthetic_test')
TRAIN_META  = os.path.join(TRAIN_DIR, 'metadata.csv')
TEST_META   = os.path.join(TEST_DIR,  'metadata.csv')

for p in [TRAIN_DIR, TEST_DIR, TRAIN_META, TEST_META]:
    status = '✓' if os.path.exists(p) else '✗ MISSING'
    print(f'{status}  {p}')